# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [1]:
import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 'sentencepiece', 'protobuf'], check=False)

import json
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from transformers import AutoTokenizer, AutoModel          
from transformers import BertTokenizer, BertModel         
from sklearn.metrics import classification_report
from collections import Counter

In [2]:
with open('./data/train-claims.json') as f:
    train_claims = json.load(f)
with open('./data/dev-claims.json') as f:
    dev_claims = json.load(f)
with open('./data/test-claims-unlabelled.json') as f:
    test_claims = json.load(f)
with open('./data/evidence.json') as f:
    evidence_corpus = json.load(f)

LABEL2IDX = {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}
IDX2LABEL  = {v: k for k, v in LABEL2IDX.items()}

print(f"Train: {len(train_claims)}  Dev: {len(dev_claims)}  Test: {len(test_claims)}")
print("Label distribution:", dict(Counter(v['claim_label'] for v in train_claims.values())))

Train: 1228  Dev: 154  Test: 153
Label distribution: {'DISPUTED': 124, 'REFUTES': 199, 'SUPPORTS': 519, 'NOT_ENOUGH_INFO': 386}


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Curated evidence pool: all evidence pieces referenced in train + dev claims.
# These ~4K pieces were hand-selected to support/refute climate claims — the same
# quality the model was trained on. Retrieving from this pool for test claims
# gives much better evidence quality than searching 1.2M general Wikipedia docs.
curated_eids = set()
for item in train_claims.values():
    curated_eids.update(item['evidences'])
for item in dev_claims.values():
    curated_eids.update(item['evidences'])

filtered_ids   = list(curated_eids)
filtered_texts = [evidence_corpus[eid] for eid in filtered_ids]
print(f"Curated evidence pool: {len(filtered_ids)} unique pieces from train+dev claims")

print("Building TF-IDF index on curated pool...")
tfidf     = TfidfVectorizer(max_features=10000, stop_words='english')
ev_matrix = tfidf.fit_transform(filtered_texts)

def retrieve_top_k(claim_text, k=5):
    q      = tfidf.transform([claim_text])
    scores = cos_sim(q, ev_matrix).flatten()
    top_k  = scores.argsort()[-k:][::-1]
    return [filtered_ids[i] for i in top_k]

def add_retrieved_evidence(claims_dict, k=5):
    return {
        cid: {**item, 'evidences': retrieve_top_k(item['claim_text'], k)}
        for cid, item in claims_dict.items()
    }

print("Retrieving evidence for train / dev / test from curated pool...")
train_claims_ret = add_retrieved_evidence(train_claims)
dev_claims_ret   = add_retrieved_evidence(dev_claims)
test_claims_ret  = add_retrieved_evidence(test_claims)
print("Done.")

Curated evidence pool: 3443 unique pieces from train+dev claims
Building TF-IDF index on curated pool...
Retrieving evidence for train / dev / test from curated pool...
Done.


In [ ]:
def get_cls_embedding(text, bert_model, tokenizer, device, max_len=128):
    """Encode a single text and return its [CLS] token embedding (shape: [768])."""
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_len, padding='max_length'
    ).to(device)
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze(0).cpu()


class ClaimEvidenceDataset(Dataset):
    """Pre-computes frozen BERT [CLS] embeddings for all claims and their evidence."""
    def __init__(self, claims_dict, evidence_corpus, label2idx, bert_model, tokenizer, device, has_labels=True):
        self.samples = []
        bert_model.eval()
        for item in claims_dict.values():
            claim_emb = get_cls_embedding(item['claim_text'], bert_model, tokenizer, device)
            ev_embs   = torch.stack([
                get_cls_embedding(evidence_corpus[eid], bert_model, tokenizer, device)
                for eid in item['evidences']
            ])  # [n_ev, 768]
            label = label2idx[item['claim_label']] if has_labels else -1
            self.samples.append((claim_emb, ev_embs, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    claim_embs, ev_embs, labels = zip(*batch)
    claim_embs = torch.stack(claim_embs)                       # [B, 768]
    lengths    = torch.tensor([e.shape[0] for e in ev_embs])   # [B]
    padded_ev  = torch.zeros(len(ev_embs), lengths.max().item(), 768)
    for i, ev in enumerate(ev_embs):
        padded_ev[i, :ev.shape[0]] = ev                        # [B, max_ev, 768]
    return claim_embs, padded_ev, lengths, torch.tensor(labels)

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [5]:
class BiLSTMClaimClassifier(nn.Module):
    """
    Architecture:
      claim [CLS] vec  ─────────────────────────────────┐
                                                         ├─ concat ─► Linear ─► 4-class label
      evidence[0..n] [CLS] vecs ─► BiLSTM ─► fwd+bwd ──┘
    """
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(768 + hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, claim_embs, ev_embs, lengths):
        # ev_embs: [B, max_ev, 768]
        packed = pack_padded_sequence(ev_embs, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.bilstm(packed)
        # h_n: [num_layers*2, B, hidden_dim] — take last layer forward + backward states
        ev_vec   = torch.cat([h_n[-2], h_n[-1]], dim=1)    # [B, hidden_dim*2]
        combined = torch.cat([claim_embs, ev_vec], dim=1)  # [B, 768 + hidden_dim*2]
        return self.classifier(combined)

In [6]:
def train_model(model, train_loader, dev_loader, device, epochs=15, lr=1e-3):
    # Inverse-frequency weights to handle class imbalance (SUPPORTS >> DISPUTED)
    counts  = Counter(s[2] for s in train_loader.dataset.samples)
    total   = sum(counts.values())
    weights = torch.tensor([total / counts[i] for i in range(4)], dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_acc, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for claim_embs, ev_embs, lengths, labels in train_loader:
            claim_embs = claim_embs.to(device)
            ev_embs    = ev_embs.to(device)
            labels     = labels.to(device)

            optimizer.zero_grad()
            logits = model(claim_embs, ev_embs, lengths)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            run_loss += loss.item()
            correct  += (logits.argmax(1) == labels).sum().item()
            total    += labels.size(0)

        dev_acc, _ = evaluate(model, dev_loader, device)
        print(f"Epoch {epoch:2d} | Loss {run_loss/len(train_loader):.4f} | "
              f"Train Acc {correct/total:.4f} | Dev Acc {dev_acc:.4f}")

        if dev_acc > best_acc:
            best_acc  = dev_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\nBest dev accuracy: {best_acc:.4f}")
    model.load_state_dict(best_state)
    return model


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for claim_embs, ev_embs, lengths, labels in loader:
            claim_embs = claim_embs.to(device)
            ev_embs    = ev_embs.to(device)
            preds = model(claim_embs, ev_embs, lengths).argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    return acc, (all_preds, all_labels)

In [7]:
class UniLSTMClaimClassifier(nn.Module):
    """
    Unidirectional baseline for comparison with BiLSTMClaimClassifier.

    Smaller parameter count 
    """
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(768 + hidden_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, claim_embs, ev_embs, lengths):
        packed = pack_padded_sequence(ev_embs, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        ev_vec   = h_n[-1]                                 
        combined = torch.cat([claim_embs, ev_vec], dim=1)   
        return self.classifier(combined)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert      = BertModel.from_pretrained('bert-base-uncased').to(device)
for param in bert.parameters():
    param.requires_grad = False
bert.eval()

# Use retrieved evidence for both train and dev — same distribution as test
print("Pre-computing BERT embeddings for train set...")
train_dataset = ClaimEvidenceDataset(train_claims_ret, evidence_corpus, LABEL2IDX, bert, tokenizer, device)
print("Pre-computing BERT embeddings for dev set...")
dev_dataset   = ClaimEvidenceDataset(dev_claims_ret,   evidence_corpus, LABEL2IDX, bert, tokenizer, device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)

model = BiLSTMClaimClassifier(
    input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
).to(device)
print(f"BiLSTM classifier parameters: {sum(p.numel() for p in model.parameters()):,}")

In [8]:
device = torch.device(
    'mps' if torch.backends.mps.is_available()  # Apple Silicon GPU
    else 'cuda' if torch.cuda.is_available() # NVIDIA GPU
    else 'cpu'
)
print(f"Using device: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert      = BertModel.from_pretrained('bert-base-uncased').to(device)
for param in bert.parameters():
    param.requires_grad = False
bert.eval()

print("Pre-computing BERT embeddings for train set (retrieved evidence from curated pool)...")
train_dataset    = ClaimEvidenceDataset(train_claims_ret, evidence_corpus, LABEL2IDX, bert, tokenizer, device)
print("Pre-computing BERT embeddings for dev set (gold evidence — upper bound)...")
dev_dataset_gold = ClaimEvidenceDataset(dev_claims,       evidence_corpus, LABEL2IDX, bert, tokenizer, device)
print("Pre-computing BERT embeddings for dev set (retrieved evidence — test-like)...")
dev_dataset_ret  = ClaimEvidenceDataset(dev_claims_ret,   evidence_corpus, LABEL2IDX, bert, tokenizer, device)

train_loader   = DataLoader(train_dataset,    batch_size=32, shuffle=True,  collate_fn=collate_fn)
dev_loader     = DataLoader(dev_dataset_gold, batch_size=32, shuffle=False, collate_fn=collate_fn)
dev_loader_ret = DataLoader(dev_dataset_ret,  batch_size=32, shuffle=False, collate_fn=collate_fn)

Using device: mps


/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Plea

Pre-computing BERT embeddings for train set (retrieved evidence from curated pool)...
Pre-computing BERT embeddings for dev set (gold evidence — upper bound)...
Pre-computing BERT embeddings for dev set (retrieved evidence — test-like)...


In [9]:
# --- Train BiLSTM ---
print("=" * 50)
print("Training BiLSTM Classifier")
print("=" * 50)
model_bilstm = BiLSTMClaimClassifier(
    input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
).to(device)

model_bilstm = train_model(model_bilstm, train_loader, dev_loader_ret, device, epochs=15, lr=1e-3)

# --- Train Unidirectional LSTM ---
print("\n" + "=" * 50)
print("Training Unidirectional LSTM Classifier")
print("=" * 50)
model_unilstm = UniLSTMClaimClassifier(
    input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
).to(device)
model_unilstm = train_model(model_unilstm, train_loader, dev_loader_ret, device, epochs=15, lr=1e-3)


bilstm_acc,  (bilstm_preds,  dev_ret_labels) = evaluate(model_bilstm,  dev_loader_ret, device)
unilstm_acc, (unilstm_preds, _)              = evaluate(model_unilstm, dev_loader_ret, device)

# Secondary metric: gold-evidence dev (upper bound)
bilstm_gold,  _ = evaluate(model_bilstm,  dev_loader, device)
unilstm_gold, _ = evaluate(model_unilstm, dev_loader, device)

print("\n" + "=" * 65)
print(f"{'Model':<10} {'Retrieved Dev':>15} {'Gold Dev':>10} {'Params':>12}")
print("-" * 65)
print(f"{'BiLSTM':<10} {bilstm_acc:>15.4f} {bilstm_gold:>10.4f} "
      f"{sum(p.numel() for p in model_bilstm.parameters()):>12,}")
print(f"{'UniLSTM':<10} {unilstm_acc:>15.4f} {unilstm_gold:>10.4f} "
      f"{sum(p.numel() for p in model_unilstm.parameters()):>12,}")
print("=" * 65)

print("\nBiLSTM classification report :")
print(classification_report(dev_ret_labels, bilstm_preds,
                             target_names=[IDX2LABEL[i] for i in range(4)], zero_division=0))

print("Unidirectional LSTM classification report :")
print(classification_report(dev_ret_labels, unilstm_preds,
                             target_names=[IDX2LABEL[i] for i in range(4)], zero_division=0))

# Forward the better model to the test prediction cell
model      = model_bilstm  if bilstm_acc >= unilstm_acc else model_unilstm
model_name = "BiLSTM"      if bilstm_acc >= unilstm_acc else "UniLSTM"
print(f"Selected model for test predictions: {model_name}")

Training BiLSTM Classifier
Epoch  1 | Loss 1.3997 | Train Acc 0.3379 | Dev Acc 0.2273
Epoch  2 | Loss 1.3440 | Train Acc 0.3664 | Dev Acc 0.3506
Epoch  3 | Loss 1.2854 | Train Acc 0.4210 | Dev Acc 0.4156
Epoch  4 | Loss 1.2706 | Train Acc 0.4121 | Dev Acc 0.3831
Epoch  5 | Loss 1.2282 | Train Acc 0.4731 | Dev Acc 0.3961
Epoch  6 | Loss 1.2200 | Train Acc 0.4381 | Dev Acc 0.3312
Epoch  7 | Loss 1.1906 | Train Acc 0.4634 | Dev Acc 0.3312
Epoch  8 | Loss 1.1598 | Train Acc 0.4845 | Dev Acc 0.4481
Epoch  9 | Loss 1.1251 | Train Acc 0.5016 | Dev Acc 0.4416
Epoch 10 | Loss 1.0817 | Train Acc 0.5293 | Dev Acc 0.4286
Epoch 11 | Loss 1.0462 | Train Acc 0.5342 | Dev Acc 0.4545
Epoch 12 | Loss 1.0433 | Train Acc 0.5651 | Dev Acc 0.4091
Epoch 13 | Loss 0.9800 | Train Acc 0.5953 | Dev Acc 0.4156
Epoch 14 | Loss 0.9473 | Train Acc 0.5912 | Dev Acc 0.4091
Epoch 15 | Loss 0.8899 | Train Acc 0.6173 | Dev Acc 0.4156

Best dev accuracy: 0.4545

Training Unidirectional LSTM Classifier
Epoch  1 | Loss 1.39

In [10]:
# ── Encoder comparison: RoBERTa-base and DeBERTa-v3-base vs BERT-base ─────────
# Each encoder is frozen; a fresh BiLSTM classifier is trained on top.
# All experiments use retrieved evidence (curated pool) to match test distribution.
# The best model by retrieved-evidence dev accuracy is forwarded to the test cell.

def build_enc_loaders(encoder, enc_tokenizer, bs=32):
    """Pre-compute frozen-encoder embeddings and return train/dev-gold/dev-ret loaders."""
    print("  -> train (retrieved)...")
    ds_tr  = ClaimEvidenceDataset(train_claims_ret, evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    print("  -> dev (gold)...")
    ds_dv  = ClaimEvidenceDataset(dev_claims,       evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    print("  -> dev (retrieved)...")
    ds_dvr = ClaimEvidenceDataset(dev_claims_ret,   evidence_corpus, LABEL2IDX,
                                   encoder, enc_tokenizer, device)
    return (DataLoader(ds_tr,  batch_size=bs, shuffle=True,  collate_fn=collate_fn),
            DataLoader(ds_dv,  batch_size=bs, shuffle=False, collate_fn=collate_fn),
            DataLoader(ds_dvr, batch_size=bs, shuffle=False, collate_fn=collate_fn))


def run_encoder_experiment(label, encoder, enc_tokenizer):
    print(f"\n{'='*55}\n  {label}\n{'='*55}")
    tr_loader, dv_loader, dvr_loader = build_enc_loaders(encoder, enc_tokenizer)
    m = BiLSTMClaimClassifier(
        input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.3
    ).to(device)
    m = train_model(m, tr_loader, dvr_loader, device, epochs=15, lr=1e-3)
    ret_acc,  _ = evaluate(m, dvr_loader, device)
    gold_acc, _ = evaluate(m, dv_loader,  device)
    return m, dv_loader, dvr_loader, ret_acc, gold_acc


results = []

# BERT-base (already trained in the previous cell)
bert_ret,  _ = evaluate(model_bilstm, dev_loader_ret, device)
bert_gold, _ = evaluate(model_bilstm, dev_loader,     device)
results.append(('BERT-base', bert_ret, bert_gold,
                model_bilstm, tokenizer, bert, dev_loader_ret))

# ── RoBERTa-base ──────────────────────────────────────────────────────────────
try:
    print("Loading roberta-base...")
    rob_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
    rob_encoder   = AutoModel.from_pretrained('roberta-base').to(device)
    for p in rob_encoder.parameters(): p.requires_grad = False
    rob_encoder.eval()
    model_rob, dev_loader_rob, dev_loader_rob_ret, rob_ret, rob_gold = \
        run_encoder_experiment("RoBERTa-base  (BiLSTM)", rob_encoder, rob_tokenizer)
    results.append(('RoBERTa-base', rob_ret, rob_gold,
                    model_rob, rob_tokenizer, rob_encoder, dev_loader_rob_ret))
except Exception as e:
    print(f"RoBERTa skipped: {e}")

# ── DeBERTa-v3-base ───────────────────────────────────────────────────────────
try:
    print("\nLoading microsoft/deberta-v3-base...")
    deb_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
    deb_encoder   = AutoModel.from_pretrained('microsoft/deberta-v3-base').to(device)
    for p in deb_encoder.parameters(): p.requires_grad = False
    deb_encoder.eval()
    model_deb, dev_loader_deb, dev_loader_deb_ret, deb_ret, deb_gold = \
        run_encoder_experiment("DeBERTa-v3-base  (BiLSTM)", deb_encoder, deb_tokenizer)
    results.append(('DeBERTa-v3-base', deb_ret, deb_gold,
                    model_deb, deb_tokenizer, deb_encoder, dev_loader_deb_ret))
except Exception as e:
    print(f"DeBERTa skipped: {e}")

print("\n" + "=" * 65)
print(f"{'Encoder':<22} {'Retrieved Dev':>15} {'Gold Dev':>10}")
print("-" * 65)
for name, ret, gold, *_ in results:
    print(f"{name:<22} {ret:>15.4f} {gold:>10.4f}")
print("=" * 65)

# Select the best encoder by retrieved-evidence dev accuracy
best_name, best_ret, best_gold, \
    best_encoder_model, best_encoder_tokenizer, best_encoder, best_dev_loader = \
    max(results, key=lambda r: r[1])

print(f"\nBest encoder: {best_name}  (retrieved dev acc = {best_ret:.4f})")


Loading roberta-base...


/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



  RoBERTa-base  (BiLSTM)
  -> train (retrieved)...
  -> dev (gold)...
  -> dev (retrieved)...
Epoch  1 | Loss 1.3951 | Train Acc 0.2891 | Dev Acc 0.1753
Epoch  2 | Loss 1.3861 | Train Acc 0.2687 | Dev Acc 0.4026
Epoch  3 | Loss 1.3828 | Train Acc 0.3208 | Dev Acc 0.3766
Epoch  4 | Loss 1.3715 | Train Acc 0.3461 | Dev Acc 0.3052
Epoch  5 | Loss 1.3669 | Train Acc 0.3746 | Dev Acc 0.4416
Epoch  6 | Loss 1.3512 | Train Acc 0.4112 | Dev Acc 0.4481
Epoch  7 | Loss 1.3580 | Train Acc 0.3909 | Dev Acc 0.4481
Epoch  8 | Loss 1.3467 | Train Acc 0.3868 | Dev Acc 0.4870
Epoch  9 | Loss 1.3372 | Train Acc 0.4218 | Dev Acc 0.5000
Epoch 10 | Loss 1.3278 | Train Acc 0.4349 | Dev Acc 0.4740
Epoch 11 | Loss 1.3129 | Train Acc 0.4503 | Dev Acc 0.4545
Epoch 12 | Loss 1.3065 | Train Acc 0.4153 | Dev Acc 0.5065
Epoch 13 | Loss 1.3142 | Train Acc 0.4202 | Dev Acc 0.3831
Epoch 14 | Loss 1.2997 | Train Acc 0.4137 | Dev Acc 0.4935
Epoch 15 | Loss 1.2871 | Train Acc 0.4397 | Dev Acc 0.4805

Best dev accuracy: 

/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



  DeBERTa-v3-base  (BiLSTM)
  -> train (retrieved)...
  -> dev (gold)...
  -> dev (retrieved)...
Epoch  1 | Loss 1.4187 | Train Acc 0.2655 | Dev Acc 0.1753
Epoch  2 | Loss 1.4028 | Train Acc 0.2329 | Dev Acc 0.2727
Epoch  3 | Loss 1.4036 | Train Acc 0.2614 | Dev Acc 0.1753
Epoch  4 | Loss 1.3920 | Train Acc 0.2321 | Dev Acc 0.4481
Epoch  5 | Loss 1.3968 | Train Acc 0.2508 | Dev Acc 0.4416
Epoch  6 | Loss 1.3915 | Train Acc 0.3396 | Dev Acc 0.2727
Epoch  7 | Loss 1.3873 | Train Acc 0.2288 | Dev Acc 0.1753
Epoch  8 | Loss 1.3871 | Train Acc 0.2712 | Dev Acc 0.2662
Epoch  9 | Loss 1.3867 | Train Acc 0.3054 | Dev Acc 0.2662
Epoch 10 | Loss 1.3933 | Train Acc 0.2459 | Dev Acc 0.2662
Epoch 11 | Loss 1.3844 | Train Acc 0.3094 | Dev Acc 0.2857
Epoch 12 | Loss 1.3887 | Train Acc 0.2134 | Dev Acc 0.2468
Epoch 13 | Loss 1.3901 | Train Acc 0.2899 | Dev Acc 0.4351
Epoch 14 | Loss 1.3871 | Train Acc 0.2866 | Dev Acc 0.3377
Epoch 15 | Loss 1.3874 | Train Acc 0.2769 | Dev Acc 0.2792

Best dev accurac

In [11]:
def encode_batch(texts, encoder, enc_tokenizer, max_len=128):
    """Encode a list of texts with any HuggingFace encoder in one forward pass."""
    inputs = enc_tokenizer(texts, return_tensors='pt', truncation=True,
                           max_length=max_len, padding=True).to(device)
    with torch.no_grad():
        out = encoder(**inputs)
    return out.last_hidden_state[:, 0, :].cpu() 


def retrieve_bert_reranked(claim_text, encoder, enc_tokenizer, k=5, n_candidates=50):
    """TF-IDF top-50 candidates from curated pool → re-rank by encoder cosine similarity → top-5."""
    q          = tfidf.transform([claim_text])
    tfidf_sc   = cos_sim(q, ev_matrix).flatten()
    cand_idx   = tfidf_sc.argsort()[-n_candidates:][::-1]
    cand_ids   = [filtered_ids[i] for i in cand_idx]
    cand_texts = [evidence_corpus[eid] for eid in cand_ids]

    claim_emb = get_cls_embedding(claim_text, encoder, enc_tokenizer, device)
    cand_embs = encode_batch(cand_texts, encoder, enc_tokenizer)
    bert_sc   = torch.nn.functional.cosine_similarity(claim_emb.unsqueeze(0), cand_embs)
    top_k     = bert_sc.argsort(descending=True)[:k].tolist()
    return [cand_ids[i] for i in top_k]


if 'best_encoder_model' not in vars():
    print("Encoder comparison not run — defaulting to BERT BiLSTM.")
    best_encoder_model     = model_bilstm
    best_encoder_tokenizer = tokenizer
    best_encoder           = bert
    best_dev_loader        = dev_loader_ret   

_, (preds, labels) = evaluate(best_encoder_model, best_dev_loader, device)
print("Dev set report (retrieved evidence — test-like distribution):")
print(classification_report(labels, preds, target_names=[IDX2LABEL[i] for i in range(4)],
                             zero_division=0))

print("\nRetrieving evidence for test claims (TF-IDF top-50 + encoder re-ranking over curated pool)...")
test_claims_with_ev = {
    cid: {**item, 'evidences': retrieve_bert_reranked(
              item['claim_text'], best_encoder, best_encoder_tokenizer)}
    for cid, item in test_claims.items()
}

test_dataset = ClaimEvidenceDataset(
    test_claims_with_ev, evidence_corpus, LABEL2IDX,
    best_encoder, best_encoder_tokenizer, device, has_labels=False
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
_, (test_preds, _) = evaluate(best_encoder_model, test_loader, device)

test_output = {}
for (cid, item), pred in zip(test_claims.items(), test_preds):
    test_output[cid] = {'claim_text': item['claim_text'], 'claim_label': IDX2LABEL[pred]}

with open('./data/test-claims-predictions.json', 'w') as f:
    json.dump(test_output, f, indent=2)

pred_dist = Counter(v['claim_label'] for v in test_output.values())
print(f"Prediction distribution: {dict(pred_dist)}")
print("Predictions saved to ./data/test-claims-predictions.json")

Dev set report (retrieved evidence — test-like distribution):
                 precision    recall  f1-score   support

       SUPPORTS       0.67      0.60      0.64        68
        REFUTES       0.52      0.41      0.46        27
NOT_ENOUGH_INFO       0.39      0.61      0.48        41
       DISPUTED       0.12      0.06      0.08        18

       accuracy                           0.51       154
      macro avg       0.43      0.42      0.41       154
   weighted avg       0.51      0.51      0.50       154


Retrieving evidence for test claims (TF-IDF top-50 + encoder re-ranking over curated pool)...
Prediction distribution: {'SUPPORTS': 45, 'NOT_ENOUGH_INFO': 63, 'DISPUTED': 16, 'REFUTES': 29}
Predictions saved to ./data/test-claims-predictions.json


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*